# Pipeline for OpenTargets Data

In [1]:
from google.cloud import bigquery
import pandas as pd
import numpy as np

PROJECT_ID = 'gene-expression-big-data'
client = bigquery.Client(project=PROJECT_ID)

# # Pull the table
# cpm_matrix = client.query("""
#     SELECT *
#     FROM `gene-expression-big-data.raw_gene_data.cpm_matrix_table`
# """).to_dataframe()

# print(cpm_matrix.shape)
# cpm_matrix.head()

In [ ]:
# DELETE
cpm = pd.read_parquet('sample_by_gene.parquet')
trait_associations = pd.read_parquet('broken_disease_associations.parquet')

gene_list = [c for c in cpm.columns if c.startswith('ENSG')]

# Re-pull associations WITH therapeutic_area_ids column
ot_query = """
SELECT
  a.targetId AS gene_id,
  dis.name AS disease_name,
  dis.id AS disease_id,
  dis.therapeuticAreas AS therapeutic_area_ids,
  a.associationScore AS association_score
FROM `open-targets-prod.platform.association_overall_direct` AS a
JOIN `open-targets-prod.platform.disease` AS dis
  ON a.diseaseId = dis.id
WHERE a.targetId IN UNNEST(@gene_list)
AND a.associationScore > 0.1
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[bigquery.ArrayQueryParameter("gene_list", "STRING", gene_list)]
)
associations = client.query(ot_query, job_config=job_config).to_dataframe()

# Collect all unique therapeutic area IDs
all_ta_ids = list(set(
    ta for sublist in trait_associations['therapeutic_area_ids'].dropna()
    for ta in sublist
))
print(f"Found {len(all_ta_ids)} unique therapeutic area codes")

# Look up their names from the disease table
ta_query = """
SELECT id, name
FROM `open-targets-prod.platform.disease`
WHERE id IN UNNEST(@ta_ids)
"""
ta_job_config = bigquery.QueryJobConfig(
    query_parameters=[bigquery.ArrayQueryParameter("ta_ids", "STRING", all_ta_ids)]
)
ta_lookup = client.query(ta_query, ta_job_config).to_dataframe()
ta_dict = dict(zip(ta_lookup['id'], ta_lookup['name']))

print("Sample lookups:")
for k, v in list(ta_dict.items())[:5]:
    print(f"  {k} → {v}")

# Rebuild therapeutic_areas with actual names
trait_associations['therapeutic_areas'] = trait_associations['therapeutic_area_ids'].apply(
    lambda ids: [ta_dict.get(i, i) for i in ids] if ids is not None and len(ids) > 0 else []
)

# Verify it worked
print("\nAfter rebuild:")
print(trait_associations[['disease_name', 'therapeutic_areas']].head(5))

# Save the fixed version
trait_associations.to_parquet('disease_associations.parquet')
print("\nSaved fixed parquet.")

Found 1 unique therapeutic area codes
Sample lookups:

After rebuild:
                          disease_name therapeutic_areas
0  B-cell acute lymphoblastic leukemia            [list]
1  B-cell acute lymphoblastic leukemia            [list]
2         chronic lymphocytic leukemia            [list]
3         chronic lymphocytic leukemia            [list]
4         chronic lymphocytic leukemia            [list]

Saved fixed parquet.


In [19]:

# Get gene list from CPM matrix
cpm = pd.read_parquet('sample_by_gene.parquet')
gene_list = [c for c in cpm.columns if c.startswith('ENSG')]
print(f"Querying for {len(gene_list)} genes")

# Pull associations WITH therapeutic_area_ids
ot_query = """
SELECT
  a.targetId AS gene_id,
  dis.name AS disease_name,
  dis.id AS disease_id,
  dis.therapeuticAreas AS therapeutic_area_ids,
  a.associationScore AS association_score
FROM `open-targets-prod.platform.association_overall_direct` AS a
JOIN `open-targets-prod.platform.disease` AS dis
  ON a.diseaseId = dis.id
WHERE a.targetId IN UNNEST(@gene_list)
AND a.associationScore > 0.1
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[bigquery.ArrayQueryParameter("gene_list", "STRING", gene_list)]
)
trait_associations = client.query(ot_query, job_config=job_config).to_dataframe()

# CRITICAL: verify therapeutic_area_ids actually came back as arrays
print(f"\nPulled {len(trait_associations)} associations")
print(f"\nFirst row therapeutic_area_ids: {trait_associations['therapeutic_area_ids'].iloc[0]}")
print(f"Type: {type(trait_associations['therapeutic_area_ids'].iloc[0])}")

Querying for 78893 genes

Pulled 705265 associations

First row therapeutic_area_ids: {'list': array([{'element': 'EFO_0005803'}, {'element': 'EFO_0000540'},
       {'element': 'OTAR_0000018'}, {'element': 'MONDO_0045024'}],
      dtype=object)}
Type: <class 'dict'>


In [21]:
def unwrap_ids(cell):
    """Extract list of EFO IDs from BQ's nested dict structure."""
    if cell is None:
        return []
    if isinstance(cell, dict) and 'list' in cell:
        return [item['element'] for item in cell['list'] if 'element' in item]
    if isinstance(cell, (list, tuple)):
        return list(cell)
    return []

# Collect unique IDs
all_ta_ids = set()
for cell in trait_associations['therapeutic_area_ids'].dropna():
    for ta in unwrap_ids(cell):
        all_ta_ids.add(ta)
all_ta_ids = list(all_ta_ids)

print(f"Found {len(all_ta_ids)} unique therapeutic area codes")
print(f"Sample: {all_ta_ids[:5]}")

ta_query = """
SELECT id, name
FROM `open-targets-prod.platform.disease`
WHERE id IN UNNEST(@ta_ids)
"""
ta_job_config = bigquery.QueryJobConfig(
    query_parameters=[bigquery.ArrayQueryParameter("ta_ids", "STRING", all_ta_ids)]
)
ta_lookup = client.query(ta_query, ta_job_config).to_dataframe()
ta_dict = dict(zip(ta_lookup['id'], ta_lookup['name']))

print(f"\nFound names for {len(ta_dict)} areas")
for k, v in list(ta_dict.items())[:5]:
    print(f"  {k} → {v}")

# Map to names using the unwrap function
trait_associations['therapeutic_areas'] = trait_associations['therapeutic_area_ids'].apply(
    lambda cell: [ta_dict.get(i, i) for i in unwrap_ids(cell)]
)

print("\nAfter mapping:")
print(trait_associations[['disease_name', 'therapeutic_areas']].head(5))

trait_associations.to_parquet('disease_associations.parquet')
print("\nSaved.")

Found 25 unique therapeutic area codes
Sample: ['EFO_0005932', 'OTAR_0000020', 'OTAR_0000018', 'GO_0008150', 'MONDO_0045024']

Found names for 25 areas
  GO_0008150 → biological_process
  MONDO_0002025 → psychiatric disorder
  MONDO_0021205 → disorder of ear
  MONDO_0024458 → disorder of visual system
  MONDO_0045024 → cancer or benign tumor

After mapping:
                          disease_name  \
0  B-cell acute lymphoblastic leukemia   
1  B-cell acute lymphoblastic leukemia   
2         chronic lymphocytic leukemia   
3         chronic lymphocytic leukemia   
4         chronic lymphocytic leukemia   

                                   therapeutic_areas  
0  [hematologic disease, immune system disease, g...  
1  [hematologic disease, immune system disease, g...  
2  [hematologic disease, immune system disease, g...  
3  [hematologic disease, immune system disease, g...  
4  [hematologic disease, immune system disease, g...  

Saved.


In [22]:
disease_associations = pd.read_parquet("disease_associations.parquet")
print(disease_associations.shape)
disease_associations.head()

(705265, 6)


,gene_id,disease_name,disease_id,therapeutic_area_ids,association_score,therapeutic_areas
0,ENSG00000104884,B-cell acute lymphoblastic leukemia,EFO_0000094,"{'list': [{'element': 'EFO_0005803'}, {'elemen...",0.277519,"[hematologic disease, immune system disease, g..."
1,ENSG00000091483,B-cell acute lymphoblastic leukemia,EFO_0000094,"{'list': [{'element': 'EFO_0005803'}, {'elemen...",0.185012,"[hematologic disease, immune system disease, g..."
2,ENSG00000136754,chronic lymphocytic leukemia,EFO_0000095,"{'list': [{'element': 'EFO_0005803'}, {'elemen...",0.277519,"[hematologic disease, immune system disease, g..."
3,ENSG00000141027,chronic lymphocytic leukemia,EFO_0000095,"{'list': [{'element': 'EFO_0005803'}, {'elemen...",0.277519,"[hematologic disease, immune system disease, g..."
4,ENSG00000110619,chronic lymphocytic leukemia,EFO_0000095,"{'list': [{'element': 'EFO_0005803'}, {'elemen...",0.277519,"[hematologic disease, immune system disease, g..."


# Analysis
Get the best number of genes because if we do like 20 then we only get a few gene associations whereas if we do all 78k then there is some random correlation ro whatever kind of like the stock data so we can figure it out using the log2FC

In [8]:
pca_df = pd.read_parquet('pca_df.parquet')
dist_summary = pd.read_csv('dist_summary.csv').set_index('sample_type')

# Identify the Pareto frontier treatments
sorted_df = dist_summary.sort_values('log2FC')
pareto_treatments = []
min_dist = float('inf')
for name, row in sorted_df.iterrows():
    if row['mean_dist'] < min_dist:
        pareto_treatments.append(name)
        min_dist = row['mean_dist']
print("Pareto frontier treatments:", pareto_treatments)

# Load CPM matrix and prepare control baseline
cpm = pd.read_parquet('sample_by_gene.parquet')
gene_cols = [c for c in cpm.columns if c.startswith('ENSG')]
control_mask = cpm['sample_type'] == 'Control'
control_means = cpm.loc[control_mask, gene_cols].mean(axis=0)

# Compute per-gene log2FC per treatment, filter to meaningfully disturbed genes
pseudocount = 0.5
LOG2FC_THRESHOLD = 1.0

off_target_results = {}
for tx in pareto_treatments:
    tx_mask = cpm['sample_type'] == tx
    tx_means = cpm.loc[tx_mask, gene_cols].mean(axis=0)
    log2fc = np.log2((tx_means + pseudocount) / (control_means + pseudocount))
    
    gene_log2fc = pd.DataFrame({
        'gene_id': gene_cols,
        'log2fc': log2fc.values,
        'abs_log2fc': np.abs(log2fc.values),
    })
    
    meaningful = gene_log2fc[gene_log2fc['abs_log2fc'] >= LOG2FC_THRESHOLD]
    off_target_results[tx] = meaningful
    print(f"{tx}: {len(meaningful)} genes with |log2FC| >= {LOG2FC_THRESHOLD}")

# Enrich with trait associations and print summary per treatment
trait_associations = pd.read_parquet('disease_associations.parquet')

for tx, top_genes in off_target_results.items():
    enriched = top_genes.merge(trait_associations, on='gene_id', how='left')
    
    print(f"\n{'='*60}")
    print(f"TREATMENT: {tx}")
    print(f"  log2FC of SNHG14: {dist_summary.loc[tx, 'log2FC']:.3f}")
    print(f"  Distance to Control: {dist_summary.loc[tx, 'mean_dist']:.1f}")
    print(f"  Total meaningfully disturbed genes: {len(top_genes)}")
    print(f"{'='*60}")
    
    top_traits = (enriched
                  .dropna(subset=['disease_name'])
                  .groupby('disease_name')
                  .agg(n_genes=('gene_id', 'nunique'),
                       avg_score=('association_score', 'mean'))
                  .sort_values(['n_genes', 'avg_score'], ascending=False)
                  .head(10))
    print(top_traits)

Pareto frontier treatments: ['nZF139', 'nZF105', 'hATF561', 'hATF567']
nZF139: 64 genes with |log2FC| >= 1.0
nZF105: 61 genes with |log2FC| >= 1.0
hATF561: 25 genes with |log2FC| >= 1.0
hATF567: 62 genes with |log2FC| >= 1.0

TREATMENT: nZF139
  log2FC of SNHG14: -1.088
  Distance to Control: 274.9
  Total meaningfully disturbed genes: 64
                                                    n_genes  avg_score
disease_name                                                          
glomerular filtration rate                                5   0.274987
neoplasm                                                  5   0.116101
body height                                               4   0.298025
serum creatinine amount                                   4   0.268486
neurodegenerative disease                                 4   0.243194
high density lipoprotein cholesterol measurement          4   0.240737
aspartate aminotransferase to alanine aminotran...        3   0.276105
aspartate aminotrans

Of the [threshold] top-disturbed genes from this treatment, [n_genes] of them have known associations with [trait]. The strongest single gene-trait link has evidence score [max_score].

In [25]:
trait_associations = pd.read_parquet('disease_associations.parquet')

for tx, top_genes in off_target_results.items():
    enriched = top_genes.merge(trait_associations, on='gene_id', how='left').dropna(subset=['disease_name'])
    exploded = enriched.explode('therapeutic_areas').dropna(subset=['therapeutic_areas'])
    
    area_summary = (exploded
                    .groupby('therapeutic_areas')
                    .agg(n_genes=('gene_id', 'nunique'),
                         n_diseases=('disease_name', 'nunique'),
                         avg_score=('association_score', 'mean'))
                    .sort_values(['n_genes', 'avg_score'], ascending=False))
    
    print(f"\n{'='*60}")
    print(f"TREATMENT: {tx}")
    print(f"  log2FC of SNHG14: {dist_summary.loc[tx, 'log2FC']:.3f}")
    print(f"  Distance to Control: {dist_summary.loc[tx, 'mean_dist']:.1f}")
    print(f"  Total meaningfully disturbed genes: {len(top_genes)}")
    print(f"  Therapeutic areas touched: {len(area_summary)}")
    print(f"{'='*60}")
    print(area_summary)


TREATMENT: nZF139
  log2FC of SNHG14: -1.088
  Distance to Control: 274.9
  Total meaningfully disturbed genes: 64
  Therapeutic areas touched: 22
                                              n_genes  n_diseases  avg_score
therapeutic_areas                                                           
measurement                                        14         105   0.253974
phenotype                                          11          46   0.296170
nervous system disease                              8          27   0.273035
cancer or benign tumor                              8         133   0.253154
genetic, familial or congenital disease             7          36   0.298130
integumentary system disease                        7          25   0.241672
urinary system disease                              7          12   0.230231
musculoskeletal or connective tissue disease        6          27   0.260735
immune system disease                               6          25   0.259012
repro

In [26]:
import plotly.graph_objects as go

# Build long-format table: one row per (treatment, therapeutic_area)
area_rows = []
for tx in pareto_treatments:
    top_genes = off_target_results[tx]
    enriched = top_genes.merge(trait_associations, on='gene_id', how='left').dropna(subset=['disease_name'])
    exploded = enriched.explode('therapeutic_areas').dropna(subset=['therapeutic_areas'])
    
    area_counts = (exploded
                   .groupby('therapeutic_areas')
                   .agg(n_genes=('gene_id', 'nunique'))
                   .reset_index())
    area_counts['treatment'] = tx
    area_rows.append(area_counts)

area_long = pd.concat(area_rows, ignore_index=True)

# Normalize by total disturbed genes per treatment
treatment_totals = {tx: len(off_target_results[tx]) for tx in pareto_treatments}
area_long['pct_of_disturbed'] = area_long.apply(
    lambda r: r['n_genes'] / treatment_totals[r['treatment']] * 100, axis=1
)

# Pivot both views - all therapeutic areas as rows
count_data = area_long.pivot_table(
    index='therapeutic_areas', columns='treatment', values='n_genes', fill_value=0
)
pct_data = area_long.pivot_table(
    index='therapeutic_areas', columns='treatment', values='pct_of_disturbed', fill_value=0
)

# Sort by total impact
order = count_data.sum(axis=1).sort_values(ascending=True).index
count_data = count_data.loc[order]
pct_data = pct_data.loc[order]

# Build figure with toggleable views
fig = go.Figure()

fig.add_trace(go.Heatmap(
    z=count_data.values, x=count_data.columns, y=count_data.index,
    colorscale='Purples',
    text=count_data.values, texttemplate='%{text}', textfont=dict(size=11),
    colorbar=dict(title='# disturbed<br>genes linked'),
    visible=True,
    hovertemplate='Treatment: %{x}<br>Area: %{y}<br>Genes: %{z}<extra></extra>',
))

fig.add_trace(go.Heatmap(
    z=pct_data.values, x=pct_data.columns, y=pct_data.index,
    colorscale='Purples',
    text=pct_data.values, texttemplate='%{text:.1f}%', textfont=dict(size=11),
    colorbar=dict(title='% of disturbed<br>genes linked'),
    visible=False,
    hovertemplate='Treatment: %{x}<br>Area: %{y}<br>Share: %{z:.1f}%<extra></extra>',
))

fig.update_layout(
    updatemenus=[dict(
        type='buttons', direction='right',
        x=0.5, xanchor='center', y=1.12, yanchor='top',
        showactive=True,
        buttons=[
            dict(label='Raw counts', method='update',
                 args=[{'visible': [True, False]},
                       {'title': 'Therapeutic Area Disruption — Raw Gene Counts'}]),
            dict(label='Normalized %', method='update',
                 args=[{'visible': [False, True]},
                       {'title': 'Therapeutic Area Disruption — % of Disturbed Genes per Treatment'}]),
        ],
    )],
    title='Therapeutic Area Disruption — Raw Gene Counts',
    xaxis_title='Treatment', yaxis_title='',
    height=700, width=950,
    template='plotly_white',
)
fig.show()

# Some notes
4 Categories:
  
Diseases — named clinical conditions (Type 2 diabetes, melanoma, Alzheimer's)
Phenotypes — observable traits or clinical signs (tall stature, abnormal kidney function, fatigue) that aren't full diseases themselves
Biological processes — molecular/cellular activities (apoptosis, inflammation, lipid metabolism)
Measurements — quantitative values from labs or studies (cholesterol level, BMI, blood pressure)

Phenotypes and Measurements and Biological process are the only ones that group a ton of others into a broad category, the diseases supposed to be inferred from the name/understanding of it

In [33]:
for area in ['disease', 'phenotype', 'biological process', 'measurement']:
    diseases_in_area = (trait_associations
        .explode('therapeutic_areas')
        .query("therapeutic_areas == @area")
        ['disease_name']
        .unique())
    
    print(f"\n{'='*60}")
    print(f"{area.upper()} — {len(diseases_in_area)} unique entries")
    print(f"{'='*60}")
    print(diseases_in_area[:30])  # first 30 to keep output readable


DISEASE — 0 unique entries
[]

PHENOTYPE — 4651 unique entries
['congestive heart failure' 'diabetes mellitus' 'hypertension'
 'infectious meningitis' 'preeclampsia' 'stroke' 'obesity'
 'morbid obesity' 'cellulitis' 'pneumonia' 'essential tremor' 'autism'
 'ankylosing spondylitis' 'deep vein thrombosis' 'angina pectoris'
 'myopathy' 'alopecia areata' 'IGA glomerulonephritis'
 'pathological myopia' 'male infertility' 'osteitis deformans'
 'sclerosing cholangitis' 'anemia (phenotype)' 'gout' 'goiter'
 'body weight gain' 'diabetes mellitus type 2 associated cataract'
 'osteoarthritis, knee' 'wet macular degeneration'
 'coronary artery calcification']

BIOLOGICAL PROCESS — 0 unique entries
[]

MEASUREMENT — 8178 unique entries
['temporal measurement' 'pulmonary function measurement'
 'clinical laboratory measurement' 'cardiovascular measurement'
 'anthropometric measurement' 'erythrocyte count' 'glucose tolerance test'
 'vital capacity' 'forced expiratory volume' 'body weights and measure